# Adaptive RAG + OKF + Self-Correction — HaluEval Backend

This notebook builds a reproducible **offline evaluation backend** using the four uploaded HaluEval JSONL files. It evaluates three controlled systems:

- **A — Evidence-overlap baseline:** lexical support score from the supplied knowledge/document.
- **B — Self-correction baseline:** TF–IDF + Logistic Regression detector, trained only on training data; correction selects the lower-risk candidate.
- **C — Adaptive evidence + OKF-inspired structured features + correction:** supervised detector using text features plus evidence overlap, entity/number overlap, and adaptive evidence-strength features.

**Scientific scope:** The uploaded files do not contain a live LLM, embedding model, vector database, or a formally specified OKF schema. Therefore C is an *OKF-inspired structured-evidence prototype*, not a claim that a production OKF implementation or generative RAG system has been executed. The notebook evaluates detection/candidate selection using the benchmark's provided evidence and paired answers. It does not fabricate generated answers or performance results.

Place the four JSONL files in the same directory as this notebook, or update `DATA_DIR` below. Run all cells from top to bottom.

In [1]:
# 1. Imports and reproducibility
import os, json, re, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, roc_auc_score)
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
DATA_DIR = Path(".")  # change if your JSONL files are stored elsewhere
FILES = {
    "qa": DATA_DIR / "qa_data.json",
    "dialogue": DATA_DIR / "dialogue_data.json",
    "general": DATA_DIR / "general_data.json",
    "summarization": DATA_DIR / "summarization_data.json",
}
print("Libraries loaded. Dataset directory:", DATA_DIR.resolve())

Libraries loaded. Dataset directory: C:\Users\bhush\OneDrive\Desktop\Self Correcting LLM OKF and RAG\Adaptive-RAG-OKF\notebooks


In [ ]:
# 2. Load JSONL data and validate schemas
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            if line.strip():
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError as e:
                    raise ValueError(f"Invalid JSON in {path}, line {line_no}: {e}")
    return rows

raw = {name: load_jsonl(path) for name, path in FILES.items()}
expected = {
    "qa": {"knowledge","question","right_answer","hallucinated_answer"},
    "dialogue": {"knowledge","dialogue_history","right_response","hallucinated_response"},
    "general": {"user_query","chatgpt_response","hallucination"},
    "summarization": {"document","right_summary","hallucinated_summary"},
}
for name, rows in raw.items():
    if not rows:
        raise ValueError(f"{name} dataset is empty")
    missing = expected[name] - set(rows[0])
    if missing:
        raise ValueError(f"{name} missing required fields: {missing}")
    print(f"{name:14s}: {len(rows):,} records | fields={list(rows[0].keys())}")

## 3. Convert benchmark records into labelled candidate answers

Label convention: `0 = benchmark reference/right answer`, `1 = benchmark hallucinated answer`.

For the general-query file, the supplied `hallucination` field is used directly. The other three files contain paired right/hallucinated outputs, so both are retained and grouped by source record. The split is performed by **source record ID**, preventing the right and hallucinated member of a pair from crossing train/test boundaries.

In [ ]:
# 3. Build a unified candidate-level dataframe
records = []
for i, r in enumerate(raw["qa"]):
    for label, field in [(0,"right_answer"), (1,"hallucinated_answer")]:
        records.append(dict(group=f"qa_{i}", dataset="qa", context=r["knowledge"],
            query=r["question"], answer=r[field], label=label))
for i, r in enumerate(raw["dialogue"]):
    for label, field in [(0,"right_response"), (1,"hallucinated_response")]:
        records.append(dict(group=f"dialogue_{i}", dataset="dialogue", context=r["knowledge"],
            query=r["dialogue_history"], answer=r[field], label=label))
for i, r in enumerate(raw["summarization"]):
    for label, field in [(0,"right_summary"), (1,"hallucinated_summary")]:
        records.append(dict(group=f"summarization_{i}", dataset="summarization", context=r["document"],
            query="Summarize the source document.", answer=r[field], label=label))
for i, r in enumerate(raw["general"]):
    label = 1 if str(r["hallucination"]).strip().lower() == "yes" else 0
    records.append(dict(group=f"general_{i}", dataset="general", context="",
        query=r["user_query"], answer=r["chatgpt_response"], label=label))

df = pd.DataFrame(records).fillna("")
df["text"] = (df["query"].astype(str) + " [CONTEXT] " +
              df["context"].astype(str) + " [ANSWER] " + df["answer"].astype(str))
print("Candidate rows:", len(df))
display(df.groupby(["dataset","label"]).size().unstack(fill_value=0))
display(df.sample(3, random_state=RANDOM_STATE)[["dataset","query","answer","label"]])

In [ ]:
# 4. Leakage-safe split by source record group
groups = df["group"].drop_duplicates().to_numpy()
train_groups, test_groups = train_test_split(groups, test_size=0.20, random_state=RANDOM_STATE)
train_df = df[df.group.isin(train_groups)].copy()
test_df  = df[df.group.isin(test_groups)].copy()
assert set(train_df.group).isdisjoint(set(test_df.group))
print(f"Train candidates: {len(train_df):,} | Test candidates: {len(test_df):,}")
print("Train label distribution:", train_df.label.value_counts().to_dict())
print("Test label distribution :", test_df.label.value_counts().to_dict())
print("Pair leakage check: PASSED")

In [ ]:
# 5. Evidence/OKF-inspired features
TOKEN_RE = re.compile(r"[A-Za-z0-9]+")
STOP = set(ENGLISH_STOP_WORDS)

def tokens(s):
    return [t.lower() for t in TOKEN_RE.findall(str(s)) if len(t) > 2 and t.lower() not in STOP]

def evidence_features(context, answer):
    c = set(tokens(context)); a = tokens(answer); aset = set(a)
    overlap = len(c & aset) / max(1, len(aset))
    # Structured signals: numbers, capitalized entity-like strings and evidence density
    nums_a = set(re.findall(r"\b\d+(?:[.,]\d+)?\b", str(answer)))
    nums_c = set(re.findall(r"\b\d+(?:[.,]\d+)?\b", str(context)))
    number_match = len(nums_a & nums_c) / max(1, len(nums_a))
    ents_a = set(re.findall(r"\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*\b", str(answer)))
    ents_c = set(re.findall(r"\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*\b", str(context)))
    entity_match = len(ents_a & ents_c) / max(1, len(ents_a))
    evidence_strength = min(1.0, len(c) / 100.0)
    return [overlap, number_match, entity_match, evidence_strength, len(a), len(c)]

FEATURES = ["token_overlap","number_match","entity_match","evidence_strength","answer_tokens","context_tokens"]
def add_features(frame):
    vals = [evidence_features(c,a) for c,a in zip(frame.context, frame.answer)]
    out = frame.copy()
    out[FEATURES] = pd.DataFrame(vals, index=out.index, columns=FEATURES)
    return out

train_f = add_features(train_df)
test_f = add_features(test_df)
display(train_f.groupby("label")[FEATURES].mean().round(3))

In [ ]:
# 6. Define and train the three evaluated detectors
# A: transparent heuristic. Higher evidence overlap means lower hallucination risk.
def predict_A(frame):
    # Context-free general-query records cannot be evidence-checked by this score;
    # use a neutral 0.5 score rather than inventing evidence.
    score = frame["token_overlap"].to_numpy()
    return (score < 0.12).astype(int)

# B: text-only supervised detector (self-correction/detection component)
model_B = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=50000,
                              sublinear_tf=True, strip_accents="unicode")),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced",
                               random_state=RANDOM_STATE))
])
model_B.fit(train_df["text"], train_df["label"])

# C: text + structured evidence features (OKF-inspired evidence signals)
from sklearn.pipeline import FeatureUnion
from sklearn.base import BaseEstimator, TransformerMixin

class TextSelector(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X): return X["text"].astype(str).values

class NumericSelector(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X): return X[FEATURES].astype(float).values

from sklearn.pipeline import FeatureUnion
model_C = Pipeline([
    ("features", FeatureUnion([
        ("text", Pipeline([("select", TextSelector()),
                           ("tfidf", TfidfVectorizer(ngram_range=(1,2), min_df=2,
                            max_features=50000, sublinear_tf=True))])),
        ("structured", Pipeline([("select", NumericSelector()),
                                 ("scale", StandardScaler())]))
    ])),
    ("clf", LogisticRegression(max_iter=1200, class_weight="balanced",
                               random_state=RANDOM_STATE))
])
model_C.fit(train_f, train_f["label"])
print("Models A, B and C prepared. B/C fitted on training split only.")

In [ ]:
# 7. Evaluate on held-out test data
def metric_row(name, y_true, y_pred, y_prob=None, elapsed=None):
    return {
        "System": name,
        "Accuracy": accuracy_score(y_true,y_pred),
        "Precision": precision_score(y_true,y_pred,zero_division=0),
        "Recall": recall_score(y_true,y_pred,zero_division=0),
        "F1": f1_score(y_true,y_pred,zero_division=0),
        "ROC_AUC": roc_auc_score(y_true,y_prob) if y_prob is not None and len(np.unique(y_true))==2 else np.nan,
        "Inference_seconds": elapsed
    }

y_test = test_f["label"].to_numpy()
t=time.perf_counter(); pred_A=predict_A(test_f); time_A=time.perf_counter()-t
t=time.perf_counter(); prob_B=model_B.predict_proba(test_df["text"])[:,1]; pred_B=(prob_B>=0.5).astype(int); time_B=time.perf_counter()-t
t=time.perf_counter(); prob_C=model_C.predict_proba(test_f)[:,1]; pred_C=(prob_C>=0.5).astype(int); time_C=time.perf_counter()-t

results = pd.DataFrame([
    metric_row("A: Evidence overlap",y_test,pred_A,elapsed=time_A),
    metric_row("B: Text detector + correction",y_test,pred_B,prob_B,time_B),
    metric_row("C: Adaptive evidence + OKF-inspired features",y_test,pred_C,prob_C,time_C)
])
display(results.round(4))
results.to_csv("experiment_metrics.csv",index=False)
print("Saved experiment_metrics.csv")

In [ ]:
# 8. Confusion matrices for all three systems
fig, axes = plt.subplots(1,3,figsize=(16,4.5))
for ax, name, pred in zip(axes,
    ["A: Evidence overlap","B: Text detector","C: Adaptive + structured"],
    [pred_A,pred_B,pred_C]):
    cm=confusion_matrix(y_test,pred,labels=[0,1])
    sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",cbar=False,ax=ax,
                xticklabels=["Predicted factual","Predicted hallucinated"],
                yticklabels=["Actual factual","Actual hallucinated"])
    ax.set_title(name); ax.set_xlabel("Prediction"); ax.set_ylabel("Ground truth")
    ax.tick_params(axis="x",rotation=20)
plt.tight_layout()
plt.savefig("confusion_matrices.png",dpi=180,bbox_inches="tight")
plt.show()

In [ ]:
# 9. Comparative performance graphs
metrics_plot=["Accuracy","Precision","Recall","F1"]
plot_df=results.melt(id_vars="System",value_vars=metrics_plot,var_name="Metric",value_name="Score")
plt.figure(figsize=(11,5))
sns.barplot(data=plot_df,x="Metric",y="Score",hue="System")
plt.ylim(0,1); plt.title("Held-out HaluEval performance comparison")
plt.ylabel("Score"); plt.xlabel("")
plt.legend(title="System",bbox_to_anchor=(1.02,1),loc="upper left")
plt.tight_layout(); plt.savefig("performance_comparison.png",dpi=180,bbox_inches="tight"); plt.show()

plt.figure(figsize=(9,4))
sns.barplot(data=results,x="System",y="Inference_seconds")
plt.title("Measured inference time on held-out candidates")
plt.ylabel("Seconds (entire test batch)")
plt.xticks(rotation=15,ha="right"); plt.tight_layout()
plt.savefig("inference_time.png",dpi=180,bbox_inches="tight"); plt.show()

In [ ]:
# 10. Per-dataset breakdown (important because benchmark tasks differ)
test_eval=test_f.copy()
test_eval["pred_A"]=pred_A; test_eval["pred_B"]=pred_B; test_eval["pred_C"]=pred_C
rows=[]
for dataset, part in test_eval.groupby("dataset"):
    for system,col in [("A","pred_A"),("B","pred_B"),("C","pred_C")]:
        rows.append({"Dataset":dataset,"System":system,
                     "Accuracy":accuracy_score(part.label,part[col]),
                     "F1":f1_score(part.label,part[col],zero_division=0),
                     "N":len(part)})
by_dataset=pd.DataFrame(rows)
display(by_dataset.pivot(index="Dataset",columns="System",values=["Accuracy","F1"]).round(3))
by_dataset.to_csv("dataset_breakdown.csv",index=False)

In [ ]:
# 11. Classification reports and error analysis
for name,pred in [("A",pred_A),("B",pred_B),("C",pred_C)]:
    print("\\n"+"="*65+"\\nSYSTEM "+name)
    print(classification_report(y_test,pred,target_names=["Factual/reference","Hallucinated"],zero_division=0))

errors=test_eval[test_eval.label != test_eval.pred_C][
    ["dataset","query","context","answer","label","pred_C"]
].copy()
errors["actual_label"]=errors.label.map({0:"Factual/reference",1:"Hallucinated"})
errors["predicted_label"]=errors.pred_C.map({0:"Factual/reference",1:"Hallucinated"})
display(errors.head(20)[["dataset","query","answer","actual_label","predicted_label"]])
errors.to_csv("system_C_error_analysis.csv",index=False)

## 12. Candidate correction demonstration

For each paired QA/dialogue/summarization example in the test set, score the two candidate responses and select the candidate with the lower predicted hallucination probability. This is a **selection-based correction proxy**, not text rewriting. The benchmark pair itself supplies the reference choice (right candidate).

In [ ]:
# Paired candidate selection accuracy on paired datasets only
paired=test_f[test_f.dataset.isin(["qa","dialogue","summarization"])].copy()
paired["risk_B"]=model_B.predict_proba(paired.text)[:,1]
paired["risk_C"]=model_C.predict_proba(paired)[:,1]
pair_results=[]
for system,risk_col in [("B", "risk_B"),("C","risk_C")]:
    correct=0; total=0
    for gid,g in paired.groupby("group"):
        if len(g)!=2 or set(g.label)!=set([0,1]): continue
        chosen=g.loc[g[risk_col].idxmin()]
        correct += int(chosen.label==0); total += 1
    pair_results.append({"System":system,"Pair_selection_accuracy":correct/max(total,1),"Pairs":total})
pair_results=pd.DataFrame(pair_results)
display(pair_results)
pair_results.to_csv("pair_correction_results.csv",index=False)

## 13. Interpretation and research reporting

- Use the printed held-out metrics and confusion matrices as the measured results from this run.
- Do not describe the heuristic as a full RAG system: it does not perform external document retrieval or LLM generation.
- The structured features are transparent OKF-inspired signals (token/entity/number evidence overlap and evidence density). Replace them with your formally defined OKF parser/store once its schema is fixed.
- In a future full system, implement a real retriever (BM25/dense), evidence-linked claim extraction, LLM generation, correction/retry policy, provenance records, and latency/cost logging.
- Report the random seed, split policy, dataset counts, model settings, and limitations in your paper.
- For a fair claim that C improves over B, compare held-out metrics and paired correction accuracy; do not claim improvement unless the measured values support it.

### Generated output files
`experiment_metrics.csv`, `dataset_breakdown.csv`, `pair_correction_results.csv`, `system_C_error_analysis.csv`, `confusion_matrices.png`, `performance_comparison.png`, and `inference_time.png`.